# Sumarização de Transcrições com GPT

Este notebook utiliza a API da OpenAI para sumarizar transcrições de vídeos do YouTube relacionados ao uso de esteroides anabolizantes.

Para cada vídeo ainda não processado, o modelo extrai informações estruturadas (tema, substâncias mencionadas, efeitos colaterais, etc.) e salva o resultado em formato JSON.

## 1. Imports e Configurações

Importações e definição dos caminhos de entrada (transcrições), saída (JSONs com resumos) e do arquivo de log.

In [1]:
import pandas as pd
import os
from openai import OpenAI
from pydantic import BaseModel
from typing import Literal
import json
from tqdm import tqdm

caminho_base_entrada = os.path.expanduser('~/gdrive/trabalho/transcriptions')
caminho_base_saida = os.path.expanduser('~/gdrive/trabalho/resumos_json')
arquivo_log = './resumos/log_resumos.txt'

## 2. IDs dos Vídeos a Processar

Carrega o CSV de predições e extrai a lista de `video_id` únicos. Apenas vídeos presentes nessa lista serão sumarizados.

In [2]:
df = pd.read_csv('modelo/prediction/final_comments_match_cleaned_pred.csv')
list_ids = (df['video_id'].unique()).tolist()

In [3]:
len(list_ids)

6742

## 3. Cliente OpenAI

Lê a chave de API a partir de um arquivo local e inicializa o cliente da OpenAI.

In [4]:
with open('resumos/api_key.txt', 'r') as f:
    api_key = f.read().split()[0]

client_gpt = OpenAI(api_key=api_key)

## 4. Esquema de Saída Estruturada (Pydantic)

Define o modelo `Resumo` com os campos que o GPT deve preencher para cada vídeo: tema, motivo de uso, substâncias, efeitos, tipo de conteúdo, público-alvo, autoridade percebida e resumo textual.

In [5]:
class Resumo(BaseModel):
    tema_principal: str
    motivo_de_uso: Literal[
        "Médico/Terapêutico", 
        "Estético/Performance", 
        "Educativo/Não Especificado"
    ]
    esteroides_anabolizantes_mencionados: list[str]
    sintomas_e_efeitos_mencionados: list[str]
    menciona_fontes_cientificas: bool
    tipo_de_conteudo: Literal[
        "Experiência Pessoal", 
        "Explicação Técnica", 
        "Entrevista", 
        "Notícia", 
        "Debate", 
        "Propaganda", 
        "Outro"
    ]
    publico_alvo: Literal[
        "Atletas/Marombeiros", 
        "Pacientes", 
        "Médicos", 
        "Público Geral", 
        "Desconhecido"
    ]
    autoridade_percebida: Literal[
        "Médico", 
        "Atleta", 
        "Influenciador", 
        "Jornalista", 
        "Desconhecido"
    ]
    resumo: str

## 5. Prompt de Sumarização

Template de prompt enviado ao GPT. Instrui o modelo a extrair as informações do esquema acima a partir da transcrição do vídeo, incluindo regras de normalização para listas de substâncias e sintomas.

In [6]:
prompt_template = '''
Você é um(a) assistente para SUMARIZAÇÃO CIENTÍFICA de vídeos do YouTube, preciso de saídas CONSISTENTES e AVALIÁVEIS.

Analise o TEXTO_FONTE e extraia as seguintes informações de forma objetiva:

- "tema_principal": O assunto principal do vídeo em 2-5 palavras.

- "motivo_de_uso": Qual é o principal CONTEXTO ou MOTIVO para o uso das substâncias discutido no vídeo?
  Classifique em APENAS UMA das seguintes categorias:
  - "Médico/Terapêutico" (Se o foco for reposição hormonal, TRT, indicação médica, tratamento de hipogonadismo, etc.)
  - "Estético/Performance" (Se o foco for ganho de massa, força, estética, "shape", "maromba", performance, competição, etc.)
  - "Educativo/Não Especificado" (Se for um vídeo apenas informativo, histórico, ou se o motivo não for claro.)
  
- "esteroides_anabolizantes_mencionados": Uma lista Python de nomes de medicamentos, anabolizantes, esteroides ou substâncias (ex: "durateston", "testosterona", "hemogenin"). Se nenhum for mencionado, retorne uma lista vazia [].

- "sintomas_e_efeitos_mencionados": Uma lista Python de sintomas ou efeitos colaterais relacionados ao uso de esteroides e anabolizantes (ex: "acne", "queda de cabelo", "insonia", "ansiedade", "ganho de força"). Se nenhum for mencionado, retorne uma lista vazia [].

- "menciona_fontes_cientificas": O texto cita fontes científicas (estudos, artigos, pesquisas, etc.)?
Responda APENAS com `True` ou `False`.

- "tipo_de_conteudo": Classifique o formato do vídeo.
Responda APENAS com uma das opções: "Experiência Pessoal", "Explicação Técnica", "Entrevista", "Notícia", "Debate", "Propaganda", "Outro".

- "publico_alvo": Infira o público-alvo principal. 
Responda APENAS com uma das opções: "Atletas/Marombeiros", "Pacientes", "Médicos", "Público Geral", "Desconhecido".

- "autoridade_percebida": Se o falante se identificar, qual sua autoridade?
Responda APENAS com uma das opções: "Médico", "Atleta", "Influenciador", "Jornalista", "Desconhecido".

- "resumo": Resuma o conteúdo abaixo em até 150 palavras, enfatizando o tema principal e as ideias centrais.
Use linguagem neutra e informativa, sem expressar opiniões pessoais.
O resumo será usado para análise semântica e agrupamento de temas.

IMPORTANTE: 
Antes de gerar as listas finais de "sintomas_e_efeitos_mencionados" e "esteroides_anabolizantes_mencionados", normalize cada item seguindo estas regras:

- Sempre usar minúsculas.
- Evitar variações sinônimas; escolha apenas uma forma padronizada para cada substância ou sintoma.
- Sempre devolver no singular.
- Para sintomas/efeitos: sempre colocar o substantivo primeiro e o adjetivo depois (ex.: "libido baixa", "força aumentada").
- Para esteroides/substâncias: padronize nomes comerciais e nomes genéricos, mas NÃO invente termos nem traduções.
- Se dois itens forem semanticamente iguais, retorne apenas um.
- Remova duplicatas mesmo que escritas de formas diferentes.

TEXTO_FONTE:
'''

## 6. Controle de Progresso (Log)

Função auxiliar que lê o arquivo de log para retornar os IDs dos vídeos já processados, evitando reprocessamento.

In [7]:
def load_log():
    with open(arquivo_log, 'r') as log_out:
        ids = log_out.read().split()
        
    return ids

## 7. Loop Principal de Sumarização

Itera sobre os arquivos de transcrição. Para cada vídeo ainda não processado (conforme o log), envia a transcrição ao GPT, salva o resultado estruturado em JSON e registra o ID no log.

In [8]:
for arquivo in tqdm(os.listdir(caminho_base_entrada)):
    nome_arquivo, _ = os.path.splitext(arquivo)
    ids_processados: str = load_log()
        
    if nome_arquivo in list_ids and nome_arquivo not in ids_processados:
        with open(os.path.join(caminho_base_entrada, arquivo), 'r') as f:
            texto = f.read()
            
            response_gpt = client_gpt.responses.parse(
                model='gpt-5-nano',
                input=prompt_template + texto,
                text_format=Resumo
            )
            
            arquivo_saida_json = os.path.join(caminho_base_saida, f'{nome_arquivo}.json')
            
            with open(arquivo_log, 'a') as log:
                log.write(nome_arquivo + '\n')
                
            with open(arquivo_saida_json, 'w', encoding='utf-8') as f_out:
                json.dump(
                    response_gpt.output_parsed.model_dump(),
                    f_out,
                    ensure_ascii=False,
                    indent=4
                )

100%|██████████| 7938/7938 [00:01<00:00, 5164.83it/s]


## 8. Verificar Progresso

Exibe quantos vídeos foram processados até o momento.

In [9]:
len(ids_processados)

6488